# matmul：从程序变换到 CPU 编译产物

对应 kickoff R02。前八个代码单元保留旧 wheel 的真实 CPU 实算与审计，带 `VERSION-SKEW`。最后两节复查另行采集的源码构建 003 归档：其生产与 native reader 已匹配固定源码，不带 VERSION-SKEW。不能把当前旧 wheel 的计算当作源码 003 重跑。

原始样本由 `matmul_probe.py` 生成，完整命令和字节指纹另存 manifest。开始前运行 `python3 -B tools/sync-environment.py check`。

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "upstream-sources.lock").exists())
HERE = ROOT / "research/jax-stack"
CAPTURE = ROOT / "artifacts/jax-stack/cpu-matmul-003"
os.environ["JAX_PLATFORMS"] = "cpu"
verification = subprocess.run([sys.executable, "-B", str(HERE / "verify_research.py")], cwd=ROOT, check=True, text=True, capture_output=True)
print(verification.stdout.strip())
results = json.loads((HERE / "cpu-results.json").read_text())
index = json.loads((HERE / "source-index.json").read_text())

{"capture_id": "cpu-matmul-003", "source_entries_checked": 134, "source_edges_checked": 40, "artifact_count": 1062, "qualifiers": ["VERSION-SKEW"]}


## 1. 固定输入与数学参考

`A[4,8]`、`W[8,6]`、`X[3,4,8]`。batch 共享 W。损失是矩阵乘结果的平方和；对 A/X 和 W 都求梯度。

单样本：`dA = 2(AW)W.T`，`dW = 2A.T(AW)`。batch 的 dW 还需沿 batch 维度求和。

In [2]:
import jax
import jax.numpy as jnp
import numpy as np

with np.load(CAPTURE / "inputs.npz", allow_pickle=False) as archive:
    A, W, X = (archive[k] for k in ("a", "w", "x"))

def matmul(a, w):
    return jnp.matmul(a, w, precision=jax.lax.Precision.HIGHEST)

batched = jax.vmap(matmul, in_axes=(0, None))
def loss(a, w):
    return jnp.sum(matmul(a, w) ** 2)
def batch_loss(x, w):
    return jnp.sum(batched(x, w) ** 2)

a64, w64, x64 = (v.astype(np.float64) for v in (A, W, X))
y64, z64 = a64 @ w64, x64 @ w64
cases = [
    ("matmul", matmul, (A, W), y64),
    ("vmap", batched, (X, W), z64),
    ("grad", jax.grad(loss, argnums=(0, 1)), (A, W), (2*y64@w64.T, 2*a64.T@y64)),
    ("jit-grad-vmap", jax.grad(batch_loss, argnums=(0, 1)), (X, W), (2*z64@w64.T, 2*np.einsum("bmk,bmn->kn", x64, z64))),
]
for name, function, args, expected in cases:
    actual = jax.jit(function)(*args)
    jax.block_until_ready(actual)
    errors = []
    for observed, reference in zip(jax.tree.leaves(actual), jax.tree.leaves(expected), strict=True):
        np.testing.assert_allclose(np.asarray(observed), reference, rtol=2e-5, atol=2e-5)
        errors.append(float(np.max(np.abs(np.asarray(observed) - reference))))
    print(name, "PASS", "maximum absolute errors:", errors)
print("Environment:", jax.__version__, jax.devices())

matmul PASS maximum absolute errors: [5.102698930059546e-08]
vmap PASS maximum absolute errors: [8.485674118929865e-08]
grad PASS maximum absolute errors: [7.892122444452809e-08, 9.822404523074368e-08]
jit-grad-vmap PASS maximum absolute errors: [1.4270093773305348e-07, 1.1280561940107958e-07]
Environment: 0.11.2.dev20260830+5832e86644 [CpuDevice(id=0)]


## 2. Jaxpr、StableHLO 与导出 HLO

下面直接调用当前 JAX 的观察接口。导出 HLO 与 backend 原始 dump 是不同观测点。共同 W 的 vmap 样本可以用一个高秩收缩表示；不能把 vmap 的 batch 数当成实际发射次数。

In [3]:
print("Jaxpr for vmap:")
print(jax.make_jaxpr(batched)(X, W))
lowered = jax.jit(batched).lower(X, W)
print("StableHLO:")
print(lowered.as_text("stablehlo"))
print("Exported HLO:")
print(lowered.as_text("hlo"))

Jaxpr for vmap:
{ lambda ; a:f32[3,4,8] b:f32[8,6]. let
    c:f32[3,4,6] = dot_general[
      dimension_numbers=(([2], [0]), ([], []))
      precision=(Precision.HIGHEST, Precision.HIGHEST)
      preferred_element_type=float32
    ] a b
  in (c,) }
StableHLO:
module @jit_matmul attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<3x4x8xf32>, %arg1: tensor<8x6xf32>) -> (tensor<3x4x6xf32> {jax.result_info = "result"}) {
    %0 = stablehlo.dot_general %arg0, %arg1, contracting_dims = [2] x [0], precision = [HIGHEST, HIGHEST] : (tensor<3x4x8xf32>, tensor<8x6xf32>) -> tensor<3x4x6xf32>
    return %0 : tensor<3x4x6xf32>
  }
}

Exported HLO:
HloModule jit_matmul, entry_computation_layout={(f32[3,4,8]{2,1,0}, f32[8,6]{1,0})->f32[3,4,6]{2,1,0}}

ENTRY main.1 {
  a.1 = f32[3,4,8]{2,1,0} parameter(0)
  w.1 = f32[8,6]{1,0} parameter(1)
  ROOT dot_general.1 = f32[3,4,6]{2,1,0} dot(a.1, w.1), lhs_contracting_dims={2}, rhs_contracting_dims={

## 3. 实际 HLO passes 与代码生成

完整 capture 使用 `--xla_dump_hlo_pass_re=.+`。字面量 `.*` 在已核对源码中会跳过未改变 HLO 的 pass；`--xla_dump_emitter_re=mlir-fusion|llvm` 另行控制 emitter 中间产物。下面的模块编号来自本次文件与 header 匹配，不把它写成跨运行常量。

In [4]:
for case in results["cases"]:
    print(case["name"], "->", case["native_module_prefix"], "pass-boundary files:", case["native_pass_boundary_count"])
    for name in case["native_pass_boundaries"][:3]:
        print(" ", name)
print("Code generation artifacts:")
for path in sorted((CAPTURE / "xla-dump").glob("*.ll")):
    print(path.name)
print("Native objects:", [item["elf"] for item in results["objects"]])

matmul -> module_0004.jit_matmul pass-boundary files: 146
  module_0004.jit_matmul.0000.async-collective.after_pipeline-start.before_async-collective-custom-call-rewriter.txt
  module_0004.jit_matmul.0001.async-collective.after_async-collective-custom-call-rewriter.before_async-collective-replacer.txt
  module_0004.jit_matmul.0002.async-collective.after_async-collective-replacer.before_pipeline-end.txt
vmap_matmul -> module_0014.jit_matmul pass-boundary files: 162
  module_0014.jit_matmul.0000.async-collective.after_pipeline-start.before_async-collective-custom-call-rewriter.txt
  module_0014.jit_matmul.0001.async-collective.after_async-collective-custom-call-rewriter.before_async-collective-replacer.txt
  module_0014.jit_matmul.0002.async-collective.after_async-collective-replacer.before_pipeline-end.txt
grad_matmul -> module_0024.jit_loss pass-boundary files: 162
  module_0024.jit_loss.0000.async-collective.after_pipeline-start.before_async-collective-custom-call-rewriter.txt
  modul

## 4. 从产物回到源码

索引记录固定 revision、输入输出、约束、调用方和被调用方。native 源码条目仍标为 SOURCE-ONLY；相同函数名或 pass 名不能消除 VERSION-SKEW。

In [5]:
selected = {"jax.matmul", "jax.dot-lowering", "jax.backend-compile", "jaxlib.compile", "xla.pass-pipeline", "xla.cpu-buffers"}
for entry in index["entries"]:
    if entry["id"] in selected:
        print(entry["id"], entry["path"], "line", entry["line"])
        print(" inputs:", entry["inputs"])
        print(" outputs:", entry["outputs"])
        print(" constraints:", entry["constraint"])
        print(" callers/callees:", entry["callers"], entry["callees"])

jax.matmul upstream/jax/jax/_src/numpy/tensor_contractions.py line 138
 inputs: 数组 lhs/rhs、precision、preferred_element_type、out_sharding
 outputs: 按广播与收缩维度定义的数组
 constraints: 检查秩、batch 维度与 sharding；构造 dimension_numbers 后调用 lax.dot_general。
 callers/callees: [] ['jax.dot-general']
jax.dot-lowering upstream/jax/jax/_src/lax/lax.py line 6264
 inputs: lowering context、MLIR lhs/rhs、dimension_numbers 和 precision
 outputs: stablehlo.dot_general 及必要转换
 constraints: 构造 DotDimensionNumbers 和 precision_config；输出仍是 MLIR value。
 callers/callees: [] []
jax.backend-compile upstream/jax/jax/_src/compiler.py line 335
 inputs: backend Client、MLIR module、设备、CompileOptions、host callbacks
 outputs: LoadedExecutable 或编译异常
 constraints: 真实 backend 与 CompileOnlyPyClient 分支不同；普通路径调用 backend.compile_and_load。
 callers/callees: [] ['jaxlib.compile']
jaxlib.compile upstream/jax/jaxlib/py_client.cc line 475
 inputs: MLIR module、设备和编译参数
 outputs: PyLoadedExecutable
 constraints: 克隆 module，包装 HloProgram/IFRT 编译选项；这是

## 5. cost、buffer assignment 与测量边界

静态 FLOPs/bytes 不是实际 DRAM 流量；compiler temporary storage 不是设备峰值。目标 roofline、split 降峰值和 overlap 仍需对应的硬件输入、运行测量与真实业务实验。

In [6]:
for case in results["cases"]:
    costs = case["cost_after_optimization"]
    memory = case["memory_analysis"]
    print(case["name"], {"flops": costs.get("flops"), "estimated_bytes_accessed": costs.get("bytes accessed"), "compiler_temp_bytes": memory.get("temp_size_in_bytes")})
print("Tracing:", json.loads((CAPTURE / "tracing.json").read_text())["observations"])
print("Separate native compilation completion events:", results["cache_compilation_completions"])

matmul {'flops': 384.0, 'estimated_bytes_accessed': 416.0, 'compiler_temp_bytes': 0}
vmap_matmul {'flops': 1152.0, 'estimated_bytes_accessed': 864.0, 'compiler_temp_bytes': 0}
grad_matmul {'flops': 1176.0, 'estimated_bytes_accessed': 1456.0, 'compiler_temp_bytes': 96}
jit_grad_vmap_matmul {'flops': 3600.0, 'estimated_bytes_accessed': 3760.0, 'compiler_temp_bytes': 608}
Tracing: [{'call': 1, 'lhs_shape': [4, 8], 'trace_count': 1}, {'call': 2, 'lhs_shape': [4, 8], 'trace_count': 1}, {'call': 3, 'lhs_shape': [5, 8], 'trace_count': 2}]
Separate native compilation completion events: 2


## 本 Notebook 的验证边界

上述单元执行旧 wheel CPU 路径。项目另已完成匹配源码构建、CPU pass Hack 的 build/load/rollback，见 `source-runtime-baseline.md` 与 `pass-hack-results.json`。下方追加归档读取，不在旧 wheel 中执行匹配源码 native reader。TPU 编译/运行、实际业务 fusion/split 与通信 overlap 仍未完成，完整清单见 `PLAN.md`。

## LLVM、ELF 与序列化包的字节对应

下面重新审计已有 CPU capture：绑定 HLO fusion、LLVM 函数、ELF global symbol，并查找完整对象在序列化包中的唯一位置。只读字节和反汇编；不 unpickle 或加载历史文件。完整调用链见 [LLVM/ORC 说明](llvm-and-objects.md)。

In [7]:
from datetime import datetime, timezone
codegen_capture = ROOT / "artifacts/jax-stack" / ("codegen-notebook-" + datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S%f"))
command = [sys.executable, "-B", str(HERE / "audit_codegen_artifacts.py"), "--output", str(codegen_capture)]
run = subprocess.run(command, cwd=ROOT, text=True, capture_output=True, timeout=180)
if run.returncode:
    raise RuntimeError(run.stdout + run.stderr)
print(run.stdout.strip())
sys.path.insert(0, str(HERE))
from verify_codegen_and_patch import verify_codegen, verify_patch
codegen = verify_codegen(codegen_capture)
for obj in codegen["objects"]:
    print(obj["case"], obj["defined_functions"], "package offset:", obj["serialized_package_byte_offset"])
print(codegen["evidence_level"], codegen["qualifiers"])

{"capture": "codegen-notebook-20260914215829619094", "objects": 3, "cases": [{"case": "matmul", "native_module_prefix": "module_0004.jit_matmul", "object_count": 0}, {"case": "vmap_matmul", "native_module_prefix": "module_0014.jit_matmul", "object_count": 0}, {"case": "grad_matmul", "native_module_prefix": "module_0024.jit_loss", "object_count": 1}, {"case": "jit_grad_vmap_matmul", "native_module_prefix": "module_0034.jit_batched_loss", "object_count": 2}]}


grad_matmul [{'name': 'broadcast_multiply_fusion', 'size_bytes': 62, 'section_index': '3'}] package offset: 5142
jit_grad_vmap_matmul [{'name': 'broadcast_multiply_fusion', 'size_bytes': 143, 'section_index': '3'}] package offset: 7921
jit_grad_vmap_matmul [{'name': 'copy_bitcast_fusion', 'size_bytes': 250, 'section_index': '3'}] package offset: 6698
REPLAY-OFFLINE ['VERSION-SKEW']


## 历史编译 pass 补丁准备记录

此单元仅验证最初补丁的独立副本应用/回滚，因此 `native_compiled` 保持 false。后续实际完成的最终补丁、25 个 C++ 测试、构建加载与回滚单列在 `pass-hack-results.json`；不覆盖历史记录。

In [8]:
prepared = verify_patch(ROOT / "artifacts/jax-stack/compiler-event-patch-002")
print(prepared["summary"]["event_name"])
print(prepared["summary"]["checks"])
print("native_compiled:", prepared["summary"]["native_compiled"])
assert prepared["summary"]["native_compiled"] is False
print("Historical wheel custom-event count:", prepared["negative_control"]["custom_event_count"])
print((HERE / "xla-hlo-pass-events.patch").read_text())

research_hlo_pass_run
{'context_apply': True, 'exact_candidate_bytes': True, 'reverse_apply': True, 'original_bytes_restored': True, 'original_source_tree_clean': True}
native_compiled: False
Historical wheel custom-event count: 0
diff --git a/xla/hlo/pass/hlo_pass_pipeline.cc b/xla/hlo/pass/hlo_pass_pipeline.cc
index f493932b44acbd73da73dfacebeb383bd06490fe..ebb7f3b2ba51888cbc7b3a2e5c2b25caa0d55ad5 100644
--- a/xla/hlo/pass/hlo_pass_pipeline.cc
+++ b/xla/hlo/pass/hlo_pass_pipeline.cc
@@ -245,7 +245,27 @@ absl::StatusOr<bool> HloPassPipeline::RunPassesInternal(
     if (!pass->IsPassPipeline()) {
       compilation_stats_->StartPass(pass_name);
     }
-    auto status_or_changed = RunHelper<HloT>(pass, hlo, execution_threads);
+    auto status_or_changed = [&]() {
+      std::optional<tsl::profiler::TraceMe> research_trace;
+      if (!pass->IsPassPipeline()) {
+        research_trace.emplace([&] {
+          return tsl::profiler::TraceMeEncode(
+              "research_hlo_pass_run",


## 匹配源码的 pass 归档

本节校验既有文件和 NumPy 参考，显示已由源码 003 native reader 解析并复查的 timeline。这里不会在当前旧 wheel 中重解析这些 HLO；完整 native 复查命令见 [逐 pass 导读](matmul-pass-walkthrough.md)。

In [9]:
import hashlib
from verify_codegen_and_patch import inventory
source_root = ROOT / "artifacts/jax-stack/source-runtime-002/suite/matmul"
pass_root = ROOT / "artifacts/jax-stack/source-lowering-audit-001/passes"
pass_result = json.loads((HERE / "pass-transition-results.json").read_text())
assert inventory(pass_root, "REPLAY-OFFLINE")["manifest_sha256"] == pass_result["manifest_sha256"]
assert pass_result["qualifiers"] == [] and pass_result["verification_runtime"]["source_build_verified"]
source_manifest = json.loads((source_root / "manifest.json").read_text())
assert hashlib.sha256((source_root / "manifest.json").read_bytes()).hexdigest() == pass_result["source_manifest_sha256"]
for item in source_manifest["artifacts"]:
    payload = (ROOT / item["path"]).read_bytes()
    assert len(payload) == item["size_bytes"] and hashlib.sha256(payload).hexdigest() == item["sha256"]
with np.load(source_root / "inputs.npz", allow_pickle=False) as inputs:
    aa, ww, xx = [inputs[k].astype(np.float64) for k in ("a", "w", "x")]
yy, zz = aa @ ww, xx @ ww
refs = {"matmul": (yy,), "vmap_matmul": (zz,), "grad_matmul": (2*yy@ww.T, 2*aa.T@yy),
        "jit_grad_vmap_matmul": (2*zz@ww.T, 2*np.einsum("bmk,bmn->kn", xx, zz))}
for case in pass_result["cases"]:
    name = case["case"]
    with np.load(source_root / name / "outputs.npz", allow_pickle=False) as outputs:
        for i, reference in enumerate(refs[name]):
            np.testing.assert_allclose(outputs[f"output_{i}"], reference, rtol=2e-5, atol=2e-5)
    timeline = json.loads((pass_root / name / "timeline.json").read_text())
    print(name, case["final_entry"])
    print([(p["sequence"], p["pass"], p["scope"]) for p in timeline["pairs"] if p["text_changed"]])
assert sum(c["numbered_pipeline_boundaries"] for c in pass_result["cases"]) == 640
print("Archive/NumPy audit passed; native source-wheel verification is a separate recorded process.")


matmul {'ynn_custom_fusions': 1, 'loop_fusions': 0, 'remaining_entry_dots': 0}
[(117, 'dot-library-rewriter', 'leaf-boundary')]
vmap_matmul {'ynn_custom_fusions': 1, 'loop_fusions': 0, 'remaining_entry_dots': 0}
[(16, 'dot_decomposer', 'leaf-boundary'), (55, 'dynamic-dimension-simplifier', 'leaf-boundary'), (64, 'algsimp', 'leaf-boundary'), (95, 'simplification', 'nested-pipeline-aggregate'), (128, 'reshape-decomposer', 'leaf-boundary'), (133, 'dot-library-rewriter', 'leaf-boundary')]
grad_matmul {'ynn_custom_fusions': 2, 'loop_fusions': 1, 'remaining_entry_dots': 1}
[(64, 'algsimp', 'leaf-boundary'), (95, 'simplification', 'nested-pipeline-aggregate'), (123, 'layout-assignment', 'leaf-boundary'), (133, 'dot-library-rewriter', 'leaf-boundary'), (138, 'fusion', 'leaf-boundary')]
jit_grad_vmap_matmul {'ynn_custom_fusions': 2, 'loop_fusions': 2, 'remaining_entry_dots': 1}
[(16, 'dot_decomposer', 'leaf-boundary'), (55, 'dynamic-dimension-simplifier', 'leaf-boundary'), (64, 'algsimp', 'leaf

## 匹配源码的 LLVM / ELF / 序列化字节

本节只读三个对象的原始字节及已有审计清单，不加载 executable。源码 capture 的包偏移与旧 capture 不同。没有对象 dump 的前向例子仍有库 runtime 路径，不能据此判断没有机器码。

In [10]:
source_codegen = json.loads((HERE / "source-codegen-results.json").read_text())
codegen_root = ROOT / "artifacts/jax-stack/source-lowering-audit-001/codegen"
assert inventory(codegen_root, "REPLAY-OFFLINE")["manifest_sha256"] == source_codegen["manifest_sha256"]
assert source_codegen["qualifiers"] == [] and source_codegen["verification_runtime"]["source_build_verified"]
from verify_codegen_and_patch import embedded_object
for obj in source_codegen["objects"]:
    object_bytes = (ROOT / obj["inputs"]["object"]).read_bytes()
    package_bytes = (ROOT / obj["inputs"]["serialized_executable"]).read_bytes()
    embedded_object(package_bytes, object_bytes, obj["serialized_package_byte_offset"])
    print(obj["case"], obj["defined_functions"], "package offset", obj["serialized_package_byte_offset"])
print("Exact-byte archive audit passed; no source native reader or executable run in this cell.")


grad_matmul [{'name': 'broadcast_multiply_fusion', 'size_bytes': 62, 'section_index': '3'}] package offset 5162
jit_grad_vmap_matmul [{'name': 'broadcast_multiply_fusion', 'size_bytes': 143, 'section_index': '3'}] package offset 6718
jit_grad_vmap_matmul [{'name': 'copy_bitcast_fusion', 'size_bytes': 250, 'section_index': '3'}] package offset 7837
Exact-byte archive audit passed; no source native reader or executable run in this cell.
